# Proyecto final · Predicción de partidos ATP
## Paso 4 · Columnas: cuáles entran, cuáles se crean y cuáles se descartan

La tabla limpia tiene lo que se sabe de cada jugador **el día del torneo** (ranking, edad, estatura, mano, siembra). Pero no dice nada de cómo viene jugando. Eso lo construimos a partir del historial de partidos.

**Regla de oro:** cada variable de un partido se calcula **solo con partidos anteriores** a ese partido. Para eso se recorre el historial en orden cronológico: primero se *anota* el estado de cada jugador y después se *actualiza* con el resultado.

| Grupo | Variables que se crean |
|---|---|
| Fuerza | Elo general y Elo por superficie |
| Experiencia | Partidos jugados en total |
| Forma | % de victorias en los últimos 10 partidos; % de victorias en la superficie |
| Historial directo | Victorias de A sobre B y de B sobre A |
| Estilo de juego | % de puntos ganados con el saque y al resto, aces, break points salvados (últimos 20 partidos) |
| Cansancio | Partidos y minutos en los últimos 14 días; días de descanso |

Al final todo se expresa como **diferencia A − B**, porque lo que decide un partido es quién es mejor, no qué tan bueno es cada uno por separado.

In [1]:
import os
import json
import time
from collections import defaultdict, deque
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

CARPETA = os.environ.get("TENIS_DATOS", "datos")
AB = ["A", "B"]

tabla = pd.read_csv(os.path.join(CARPETA, "tabla_limpia.csv.gz"), low_memory=False, parse_dates=["fecha"])
hist = pd.read_csv(os.path.join(CARPETA, "historial_limpio.csv.gz"), low_memory=False, parse_dates=["tourney_date"])
columnas_antes = tabla.columns.tolist()
print(f"Tabla limpia: {tabla.shape[0]:,} partidos × {tabla.shape[1]} columnas")
print(f"Historial limpio: {hist.shape[0]:,} partidos")

Tabla limpia: 78,815 partidos × 61 columnas
Historial limpio: 303,450 partidos


### Orden cronológico
Todos los partidos de un torneo tienen la misma fecha (el lunes de inicio). Para saber qué partido fue antes, dentro de cada fecha se ordena por **ronda**: qualys → primera ronda → … → final.

In [2]:
ORDEN_RONDA = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4, "ER": 5, "R128": 6, "R64": 7, "R32": 8, "R16": 9,
               "RR": 10, "QF": 11, "SF": 12, "BR": 13, "F": 14}
hist["orden_ronda"] = hist["round"].map(ORDEN_RONDA).fillna(8)
hist = hist.sort_values(["tourney_date", "orden_ronda", "tourney_id", "match_num"]).reset_index(drop=True)
print("Rondas sin orden conocido:", sorted(set(hist["round"].dropna()) - set(ORDEN_RONDA)))

Rondas sin orden conocido: []


### El Elo
Cada jugador arranca con 1.500 puntos. Después de cada partido, el ganador sube y el perdedor baja, más o menos según qué tan **sorpresivo** fue el resultado:

$$P(\text{gana } A) = \frac{1}{1 + 10^{(Elo_B - Elo_A)/400}} \qquad Elo_A \leftarrow Elo_A + K\,(\text{resultado} - P)$$

Si un favorito gana, sube poco; si pierde, baja mucho. El factor $K = 250 / (n + 5)^{0.4}$ (el de FiveThirtyEight) hace que los jugadores con pocos partidos ($n$) se ajusten rápido y los veteranos, despacio.

El **Elo por superficie** es lo mismo, pero contando solo los partidos en esa superficie: hay jugadores mucho mejores en arcilla que en pasto.

A diferencia del ranking ATP, el Elo tiene en cuenta **a quién** le ganaste, no solo cuántos puntos sumaste.

In [3]:
def factor_k(n):
    return 250 / ((n + 5) ** 0.4)

# Columnas del historial como arreglos (el ciclo es mucho más rápido así)
N = len(hist)
W, L = hist["winner_id"].values, hist["loser_id"].values
DIA = hist["tourney_date"].values.astype("datetime64[D]").astype(np.int64)
SUP = hist["surface"].values
MIN = hist["minutes"].values.astype(float)
VALIDAS = hist["stats_validas"].values.astype(bool)
st = {c: np.nan_to_num(hist[c].values.astype(float)) for p in ["w_", "l_"]
      for c in [f"{p}svpt", f"{p}1stWon", f"{p}2ndWon", f"{p}ace", f"{p}bpSaved", f"{p}bpFaced"]}

VARS = ["elo", "elo_sup", "experiencia", "forma10", "pct_sup", "saque", "resto", "ace", "bp_salv",
        "n_stats", "partidos14", "minutos14", "dias_desc", "h2h", "elo_post"]
pre = {f"{lado}_{v}": np.full(N, np.nan) for lado in ["w", "l"] for v in VARS}

# Estado de cada jugador, que se va actualizando
elo = defaultdict(lambda: 1500.0)
n_part = defaultdict(int)
elo_s = defaultdict(lambda: 1500.0)          # llave: (jugador, superficie)
n_s = defaultdict(int)
ult10 = defaultdict(lambda: deque(maxlen=10))
sup50 = defaultdict(lambda: deque(maxlen=50))
stats20 = defaultdict(lambda: deque(maxlen=20))
fechas = defaultdict(lambda: deque(maxlen=20))
h2h = defaultdict(int)                       # llave: (a, b) = victorias de a sobre b

def anotar(i, lado, p, o, d, s):
    """Guarda el estado del jugador p ANTES del partido i."""
    pre[f"{lado}_elo"][i] = elo[p]
    pre[f"{lado}_experiencia"][i] = n_part[p]
    q = ult10[p]
    pre[f"{lado}_forma10"][i] = (sum(q) + 1) / (len(q) + 2)          # suavizado: sin partidos = 50 %
    if isinstance(s, str):
        pre[f"{lado}_elo_sup"][i] = elo_s[(p, s)]
        q = sup50[(p, s)]
        pre[f"{lado}_pct_sup"][i] = (sum(q) + 1) / (len(q) + 2)
    else:
        pre[f"{lado}_elo_sup"][i] = elo[p]
        pre[f"{lado}_pct_sup"][i] = 0.5
    q = stats20[p]
    pre[f"{lado}_n_stats"][i] = len(q)
    if len(q) >= 5:                                                  # con menos de 5 partidos no es confiable
        a = np.array(q).sum(axis=0)
        pre[f"{lado}_saque"][i] = a[0] / a[1]
        pre[f"{lado}_resto"][i] = a[2] / a[3]
        pre[f"{lado}_ace"][i] = a[4] / a[1]
        pre[f"{lado}_bp_salv"][i] = (a[5] + 1) / (a[6] + 2)
    f = fechas[p]
    pre[f"{lado}_partidos14"][i] = sum(1 for dd, _ in f if d - dd <= 14)
    pre[f"{lado}_minutos14"][i] = sum(m for dd, m in f if d - dd <= 14 and m == m)
    pre[f"{lado}_dias_desc"][i] = min(d - f[-1][0], 365) if f else 365
    pre[f"{lado}_h2h"][i] = h2h[(p, o)]

inicio = time.time()
for i in range(N):
    w, l, d, s = W[i], L[i], DIA[i], SUP[i]
    anotar(i, "w", w, l, d, s)
    anotar(i, "l", l, w, d, s)

    # --- actualizar con el resultado ---
    p_w = 1 / (1 + 10 ** ((elo[l] - elo[w]) / 400))
    elo[w] += factor_k(n_part[w]) * (1 - p_w)
    elo[l] -= factor_k(n_part[l]) * (1 - p_w)
    pre["w_elo_post"][i], pre["l_elo_post"][i] = elo[w], elo[l]     # solo para demostrar una fuga
    n_part[w] += 1
    n_part[l] += 1
    if isinstance(s, str):
        kw, kl = (w, s), (l, s)
        p_ws = 1 / (1 + 10 ** ((elo_s[kl] - elo_s[kw]) / 400))
        elo_s[kw] += factor_k(n_s[kw]) * (1 - p_ws)
        elo_s[kl] -= factor_k(n_s[kl]) * (1 - p_ws)
        n_s[kw] += 1
        n_s[kl] += 1
        sup50[kw].append(1)
        sup50[kl].append(0)
    ult10[w].append(1)
    ult10[l].append(0)
    if VALIDAS[i]:
        sv_w = st["w_1stWon"][i] + st["w_2ndWon"][i]
        sv_l = st["l_1stWon"][i] + st["l_2ndWon"][i]
        stats20[w].append((sv_w, st["w_svpt"][i], st["l_svpt"][i] - sv_l, st["l_svpt"][i],
                           st["w_ace"][i], st["w_bpSaved"][i], st["w_bpFaced"][i]))
        stats20[l].append((sv_l, st["l_svpt"][i], st["w_svpt"][i] - sv_w, st["w_svpt"][i],
                           st["l_ace"][i], st["l_bpSaved"][i], st["l_bpFaced"][i]))
    fechas[w].append((d, MIN[i]))
    fechas[l].append((d, MIN[i]))
    h2h[(w, l)] += 1

print(f"{N:,} partidos recorridos en {time.time() - inicio:.0f} s; {len(elo):,} jugadores con Elo")

303,450 partidos recorridos en 33 s; 9,038 jugadores con Elo


### ¿El Elo tiene sentido?
Antes de usarlo, una prueba de cordura: los jugadores con el Elo más alto al final del historial deberían ser los mejores del momento.

In [4]:
nombres = pd.concat([tabla[["A_id", "A_name"]].set_axis(["id", "nombre"], axis=1),
                     tabla[["B_id", "B_name"]].set_axis(["id", "nombre"], axis=1)]).drop_duplicates("id").set_index("id")["nombre"]
ultimo = pd.concat([pd.DataFrame({"id": W, "dia": DIA}), pd.DataFrame({"id": L, "dia": DIA})]).groupby("id")["dia"].max()
activos = ultimo[ultimo >= DIA.max() - 180].index          # jugaron en los últimos 6 meses del historial

top = (pd.DataFrame({"id": list(elo.keys()), "Elo": list(elo.values())})
       .query("id in @activos").nlargest(10, "Elo"))
top["Jugador"] = top["id"].map(nombres)
for s in ["Hard", "Clay", "Grass"]:
    top[f"Elo {s}"] = [round(elo_s[(p, s)]) for p in top["id"]]
top["Elo"] = top["Elo"].round()
top[["Jugador", "Elo", "Elo Hard", "Elo Clay", "Elo Grass"]].reset_index(drop=True)

,Jugador,Elo,Elo Hard,Elo Clay,Elo Grass
0,Jannik Sinner,2658.0,2590,2442,2091
1,Carlos Alcaraz,2586.0,2463,2471,2196
2,Novak Djokovic,2472.0,2431,2353,2254
3,Alexander Zverev,2408.0,2313,2320,1893
4,Arthur Fils,2345.0,2217,2248,1813
5,Jack Draper,2312.0,2265,2106,1859
6,Rafael Jodar,2300.0,2066,2206,1500
7,Daniil Medvedev,2294.0,2272,2117,1917
8,Casper Ruud,2283.0,2112,2272,1582
9,Alex De Minaur,2281.0,2246,2113,1911


### Pasar las variables a la tabla principal
Cada partido del tour está en el historial con la misma llave (`tourney_id` + `match_num`). Allí las variables están como ganador / perdedor; se pasan a A / B según quién ganó.

In [5]:
hist["match_id"] = hist["tourney_id"].astype(str) + "-" + hist["match_num"].astype(str)
pos = pd.Series(hist.index, index=hist["match_id"])
pos = pos[(hist["origen"] == "tour").values]
pos = pos[~pos.index.duplicated()]

idx = tabla["match_id"].map(pos)
print(f"Partidos de la tabla encontrados en el historial: {idx.notna().mean():.2%}")
tabla = tabla[idx.notna()].reset_index(drop=True)
idx = idx.dropna().astype(int).values

gana = tabla["gana_A"].values == 1
for v in VARS:
    w_v, l_v = pre[f"w_{v}"][idx], pre[f"l_{v}"][idx]
    tabla[f"A_{v}"] = np.where(gana, w_v, l_v)
    tabla[f"B_{v}"] = np.where(gana, l_v, w_v)

# Comprobación: el jugador A del historial es el mismo de la tabla
id_hist_A = np.where(gana, W[idx], L[idx])
print(f"Coincidencia del jugador A: {(id_hist_A == tabla['A_id'].values).mean():.2%}")

Partidos de la tabla encontrados en el historial: 100.00%
Coincidencia del jugador A: 100.00%


### Construir las columnas del modelo
Casi todas como diferencia A − B. Algunas transformaciones:
- **Ranking y puntos en logaritmo:** la distancia entre el 1 y el 10 importa mucho más que entre el 200 y el 210.
- **Experiencia en logaritmo**, por la misma razón.
- **Historial directo** como diferencia de victorias y total de enfrentamientos.
- **Tipo de entrada** como marcas: viene de la qualy (Q o LL) o tiene invitación (WC).

In [6]:
t = tabla
t["dif_elo"] = t["A_elo"] - t["B_elo"]
t["dif_elo_sup"] = t["A_elo_sup"] - t["B_elo_sup"]
t["dif_log_rank"] = np.log(t["A_rank"]) - np.log(t["B_rank"])
t["dif_log_puntos"] = np.log1p(t["A_rank_points"]) - np.log1p(t["B_rank_points"])
t["dif_log_experiencia"] = np.log1p(t["A_experiencia"]) - np.log1p(t["B_experiencia"])
t["dif_forma10"] = t["A_forma10"] - t["B_forma10"]
t["dif_pct_sup"] = t["A_pct_sup"] - t["B_pct_sup"]
t["h2h_dif"] = t["A_h2h"] - t["B_h2h"]
t["h2h_total"] = t["A_h2h"] + t["B_h2h"]
t["dif_saque"] = t["A_saque"] - t["B_saque"]
t["dif_resto"] = t["A_resto"] - t["B_resto"]
t["dif_ace"] = t["A_ace"] - t["B_ace"]
t["dif_bp_salv"] = t["A_bp_salv"] - t["B_bp_salv"]
t["dif_partidos14"] = t["A_partidos14"] - t["B_partidos14"]
t["dif_minutos14"] = t["A_minutos14"] - t["B_minutos14"]
t["dif_dias_desc"] = t["A_dias_desc"] - t["B_dias_desc"]
t["dif_edad"] = t["A_age"] - t["B_age"]
t["dif_estatura"] = t["A_ht"] - t["B_ht"]
for L in AB:
    t[f"{L}_zurdo"] = (t[f"{L}_hand"] == "L").astype(int)
    t[f"{L}_qualy"] = t[f"{L}_entry"].isin(["Q", "LL"]).astype(int)
    t[f"{L}_wc"] = (t[f"{L}_entry"] == "WC").astype(int)

NUMERICAS = ["dif_elo", "dif_elo_sup", "dif_log_rank", "dif_log_puntos", "dif_log_experiencia",
             "dif_forma10", "dif_pct_sup", "h2h_dif", "h2h_total",
             "dif_saque", "dif_resto", "dif_ace", "dif_bp_salv",
             "dif_partidos14", "dif_minutos14", "dif_dias_desc", "dif_edad", "dif_estatura",
             "A_zurdo", "B_zurdo", "A_sin_ranking", "B_sin_ranking", "A_sembrado", "B_sembrado",
             "A_qualy", "B_qualy", "A_wc", "B_wc", "best_of"]
CATEGORICAS = ["superficie", "nivel", "ronda"]
print(f"Columnas del modelo: {len(NUMERICAS)} numéricas + {len(CATEGORICAS)} categóricas")
print("\nFaltantes en las columnas del modelo (%):")
falt = (t[NUMERICAS + CATEGORICAS].isna().mean() * 100).round(1)
print(falt[falt > 0].to_string() if (falt > 0).any() else "ninguno")

Columnas del modelo: 29 numéricas + 3 categóricas

Faltantes en las columnas del modelo (%):
dif_saque      7.5
dif_resto      7.5
dif_ace        7.5
dif_bp_salv    7.5


Las variables de saque y resto quedan vacías cuando un jugador tiene menos de 5 partidos con estadísticas (sobre todo en Copa Davis y en los primeros años). **No se imputan aquí**: en el paso 5 se imputan **dentro del Pipeline**, con los datos de entrenamiento, y se agrega una marca de "faltante" para que el modelo sepa que no había dato.

### ¿Cuánto predice cada variable por sí sola?
Para cada variable se mide el **AUC**: la probabilidad de que, tomando un partido que ganó A y otro que perdió, la variable sea mayor en el que ganó. 0,5 = no sirve; 1 = predice perfecto.

También se mide la exactitud de la regla "gana el que tenga más" (en el ranking, "gana el que tenga menos"). Se excluye el año 2000 para que el historial tenga tiempo de acumularse.

In [7]:
eval_ = t[t["anio"] >= 2001]
y = eval_["gana_A"]
filas = []
for c in [c for c in NUMERICAS if c.startswith(("dif_", "h2h_dif"))]:
    x = eval_[c]
    ok = x.notna()
    if ok.sum() < 50 or x[ok].nunique() < 2:
        continue
    auc = roc_auc_score(y[ok], x[ok])
    signo = 1 if auc >= 0.5 else -1                       # el ranking va al revés: menos es mejor
    pred = np.where(signo * x[ok] > 0, 1, np.where(signo * x[ok] < 0, 0, 1))
    filas.append([c, round(max(auc, 1 - auc), 3), round((pred == y[ok]).mean() * 100, 1),
                  round(ok.mean() * 100, 1), "mayor" if signo == 1 else "menor"])

poder = (pd.DataFrame(filas, columns=["Variable", "AUC", "Exactitud regla (%)", "Cobertura (%)", "Gana el de"])
         .sort_values("AUC", ascending=False).reset_index(drop=True))

# Alarma de fuga: en tenis, ninguna variable honesta predice tan bien por sí sola
sospechosas = poder.loc[poder["AUC"] > 0.85, "Variable"].tolist()
print("⚠ POSIBLE FUGA, revisar:" if sospechosas else "Sin variables sospechosas de fuga (todas con AUC ≤ 0,85).", sospechosas or "")
poder

Sin variables sospechosas de fuga (todas con AUC ≤ 0,85). 


,Variable,AUC,Exactitud regla (%),Cobertura (%),Gana el de
0,dif_elo,0.741,67.2,100.0,mayor
1,dif_elo_sup,0.738,67.2,100.0,mayor
2,dif_log_rank,0.720,65.8,100.0,menor
3,dif_log_puntos,0.720,65.8,100.0,mayor
4,dif_pct_sup,0.667,61.7,100.0,mayor
5,dif_saque,0.641,60.0,92.8,mayor
6,dif_forma10,0.611,57.8,100.0,mayor
7,dif_log_experiencia,0.593,56.4,100.0,mayor
8,h2h_dif,0.579,54.6,100.0,mayor
9,dif_bp_salv,0.574,55.3,92.8,mayor


### Una fuga a propósito
¿Qué pasa si, por error, se usa el Elo **después** del partido (que ya incluye el resultado)? Es el mismo error que `decision_renovacion` en el ejemplo de seguros.

In [8]:
fuga = eval_["A_elo_post"] - eval_["B_elo_post"]
print(f"AUC del Elo ANTES del partido:   {roc_auc_score(y, eval_['dif_elo']):.3f}")
print(f"AUC del Elo DESPUÉS del partido: {roc_auc_score(y, fuga):.3f}   ← fuga: el resultado ya está adentro")
tabla = tabla.drop(columns=["A_elo_post", "B_elo_post"])
t = tabla

AUC del Elo ANTES del partido:   0.741
AUC del Elo DESPUÉS del partido: 0.813   ← fuga: el resultado ya está adentro


### ¿Hay variables redundantes?
Si dos variables miden lo mismo, la correlación entre ellas es alta. No hace falta quitarlas para los árboles, pero en la regresión logística pueden volver inestables los coeficientes; se tiene en cuenta en el paso 5.

In [9]:
t[["dif_elo", "dif_elo_sup", "dif_log_rank", "dif_log_puntos", "dif_forma10", "dif_pct_sup", "dif_saque"]].corr().round(2)

,dif_elo,dif_elo_sup,dif_log_rank,dif_log_puntos,dif_forma10,dif_pct_sup,dif_saque
dif_elo,1.00,0.85,-0.87,0.79,0.45,0.57,0.52
dif_elo_sup,0.85,1.00,-0.73,0.65,0.40,0.79,0.45
dif_log_rank,-0.87,-0.73,1.00,-0.91,-0.30,-0.47,-0.43
dif_log_puntos,0.79,0.65,-0.91,1.00,0.26,0.40,0.40
dif_forma10,0.45,0.40,-0.30,0.26,1.00,0.42,0.35
dif_pct_sup,0.57,0.79,-0.47,0.40,0.42,1.00,0.35
dif_saque,0.52,0.45,-0.43,0.40,0.35,0.35,1.00


### Antes y después: qué columnas entran, cuáles se crean y cuáles se descartan

In [10]:
entran = ["superficie", "nivel", "ronda", "best_of", "A_sin_ranking", "B_sin_ranking",
          "A_sembrado", "B_sembrado"]
transformadas = {"A_rank / B_rank": "dif_log_rank", "A_rank_points / B_rank_points": "dif_log_puntos",
                 "A_age / B_age": "dif_edad", "A_ht / B_ht": "dif_estatura",
                 "A_hand / B_hand": "A_zurdo, B_zurdo", "A_entry / B_entry": "A_qualy, B_qualy, A_wc, B_wc"}
creadas = ["dif_elo", "dif_elo_sup", "dif_log_experiencia", "dif_forma10", "dif_pct_sup", "h2h_dif", "h2h_total",
           "dif_saque", "dif_resto", "dif_ace", "dif_bp_salv", "dif_partidos14", "dif_minutos14", "dif_dias_desc"]
descartadas = {
    "Fuga (posterior al partido)": ["score", "minutos", "tipo_final", "termino_anormal", "estadísticas del partido (18)"],
    "Identificadores y texto": ["match_id", "tourney_id", "torneo", "A_id", "B_id", "A_name", "B_name"],
    "Ya usadas o redundantes": ["fecha", "anio", "cuadro", "A_seed / B_seed", "A_ioc / B_ioc", "A_nacimiento / B_nacimiento",
                                "A_ht_imputada / B_ht_imputada"],
}
print(f"Columnas antes: {len(columnas_antes)}  →  columnas del modelo: {len(NUMERICAS) + len(CATEGORICAS)}\n")
print("ENTRAN tal cual:", ", ".join(entran))
print("\nENTRAN transformadas:")
for a, b in transformadas.items():
    print(f"   {a}  →  {b}")
print(f"\nSE CREAN ({len(creadas)}):", ", ".join(creadas))
print("\nSE DESCARTAN:")
for k, v in descartadas.items():
    print(f"   {k}: {', '.join(v)}")
print("\n(A_seed se descarta porque A_sembrado ya dice si fue cabeza de serie; el país, porque no sabemos dónde se juega cada torneo.)")

Columnas antes: 61  →  columnas del modelo: 32

ENTRAN tal cual: superficie, nivel, ronda, best_of, A_sin_ranking, B_sin_ranking, A_sembrado, B_sembrado

ENTRAN transformadas:
   A_rank / B_rank  →  dif_log_rank
   A_rank_points / B_rank_points  →  dif_log_puntos
   A_age / B_age  →  dif_edad
   A_ht / B_ht  →  dif_estatura
   A_hand / B_hand  →  A_zurdo, B_zurdo
   A_entry / B_entry  →  A_qualy, B_qualy, A_wc, B_wc

SE CREAN (14): dif_elo, dif_elo_sup, dif_log_experiencia, dif_forma10, dif_pct_sup, h2h_dif, h2h_total, dif_saque, dif_resto, dif_ace, dif_bp_salv, dif_partidos14, dif_minutos14, dif_dias_desc

SE DESCARTAN:
   Fuga (posterior al partido): score, minutos, tipo_final, termino_anormal, estadísticas del partido (18)
   Identificadores y texto: match_id, tourney_id, torneo, A_id, B_id, A_name, B_name
   Ya usadas o redundantes: fecha, anio, cuadro, A_seed / B_seed, A_ioc / B_ioc, A_nacimiento / B_nacimiento, A_ht_imputada / B_ht_imputada

(A_seed se descarta porque A_sembrado 

### Guardar la tabla del modelo

In [11]:
guardar = ["match_id", "fecha", "anio", "torneo", "A_name", "B_name", "A_rank", "B_rank", "gana_A"] + NUMERICAS + CATEGORICAS
tabla[guardar].to_csv(os.path.join(CARPETA, "tabla_modelo.csv.gz"), index=False)
with open(os.path.join(CARPETA, "columnas_modelo.json"), "w") as f:
    json.dump({"numericas": NUMERICAS, "categoricas": CATEGORICAS}, f, indent=1)
print(f"Guardada tabla_modelo.csv.gz: {len(tabla):,} partidos × {len(guardar)} columnas")

Guardada tabla_modelo.csv.gz: 78,815 partidos × 41 columnas


### Conclusión del paso 4
 
Se recorrieron todos los partidos del historial (tour, qualys y challengers desde 1995) en orden cronológico. Antes de cada partido se anota el estado de cada jugador y después se actualiza con el resultado, así que **ninguna variable usa información del mismo partido ni del futuro**.
 
**¿El Elo tiene sentido?** El top 10 al final del historial es Sinner, Alcaraz, Djokovic, Zverev, Fils, Draper, Jódar, Medvedev, Ruud y De Miñaur. El Elo por superficie reproduce lo conocido: Djokovic tiene el mejor Elo en pasto (2.254), Alcaraz en arcilla (2.471) y Sinner en dura (2.590); Ruud es mucho más fuerte en arcilla (2.272) que en pasto (1.582). Limitación: un jugador sin partidos en una superficie queda con el Elo inicial (Jódar: 1.500 en pasto).
 
**Poder predictivo de cada variable por sí sola (2001–2026)**
 
| Variable | AUC | Exactitud "gana el de mayor" |
|---|---|---|
| dif_elo | 0,741 | 67,2 % |
| dif_elo_sup | 0,738 | 67,2 % |
| dif_log_rank | 0,720 | 65,8 % |
| dif_log_puntos | 0,720 | 65,8 % |
| dif_pct_sup | 0,667 | 61,7 % |
| dif_saque | 0,641 | 60,0 % |
| dif_forma10 | 0,611 | 57,8 % |
| dif_log_experiencia | 0,593 | 56,4 % |
| h2h_dif | 0,579 | 54,6 % |
| dif_bp_salv / dif_resto / dif_ace | 0,56–0,57 | 54–55 % |
| dif_estatura, dif_edad, cansancio | 0,51–0,54 | 50–52 % |
 
- **El Elo supera al ranking antes de cualquier modelo:** 67,2 % contra 65,8 %. Tiene en cuenta *a quién* le ganó cada jugador, no solo cuántos puntos sumó.
- **Las variables de saque cubren el 92,8 %** de los partidos; el resto se imputa dentro del Pipeline en el paso 5.
- **El cansancio no predice por sí solo** (AUC ≈ 0,51), y el que jugó más en los últimos 14 días gana un poco más. No es que el cansancio ayude: quien llega a rondas avanzadas jugó más porque viene ganando. Se conserva por si aporta combinado con otras variables (por ejemplo, en partidos a 5 sets).
- **Ninguna variable supera un AUC de 0,85**, así que no hay señales de fuga.
 
**Una fuga a propósito:** si se usa el Elo *después* del partido (que ya incluye el resultado), el AUC sube de **0,741 a 0,813**. Es el mismo error que `decision_renovacion` en el ejemplo de seguros.
 
**Redundancia:** ranking y puntos están casi perfectamente correlacionados (−0,91), igual que Elo general y Elo por superficie (0,85). A los árboles no les afecta, pero en la regresión logística se dejará solo una variable de cada par.
 
**Columnas:** de 61 columnas en la tabla limpia se pasa a **32 columnas de modelo** (29 numéricas + 3 categóricas): 8 entran tal cual, 6 se transforman (logaritmos y marcas), 14 se crean desde el historial, y se descartan las de fuga, los identificadores y las redundantes.